# Exercise 2 — generate_case_study

A case study is what turns a GitHub repo into a portfolio piece. It tells the story: what problem you solved, what you built, which technologies you used, and what you achieved. `generate_case_study` produces a Markdown document with structured sections that can be hosted as a GitHub Gist, a blog post, or a page on your portfolio site.

In [ ]:
import pathlib, tempfile, os
from collections import Counter
from dataclasses import dataclass, field

@dataclass
class ProjectEntry:
    name: str; tagline: str; description: str; tech_stack: list
    github_url: str = ""; demo_url: str = ""; category: str = "AI Engineering"
    highlights: list = field(default_factory=list)

@dataclass
class PortfolioConfig:
    owner_name: str; title: str; bio: str; email: str; github_username: str
    linkedin_url: str = ""; projects: list = field(default_factory=list)

_P1 = ProjectEntry(
    name        = "AI Trading Bot",
    tagline     = "Paper-trading bot with sentiment + technical signals.",
    description = "Built over Days 89-96, this bot fetches OHLCV data, computes "
                  "technical indicators, scores news headlines with an LLM, applies "
                  "stop-loss and drawdown controls, and logs results daily.",
    tech_stack  = ["Python", "pandas", "Ollama", "SQLite"],
    github_url  = "https://github.com/testuser/ai-trading-bot",
    category    = "Finance",
    highlights  = ["Fully automated daily paper-trading loop",
                   "Kelly Criterion position sizing", "Stop-loss + drawdown gating"],
)
_P2 = ProjectEntry(
    name        = "Ops Agent",
    tagline     = "Autonomous multi-step ops agent with guardrails.",
    description = "Agent loop with tool routing, human-in-the-loop approval gates, "
                  "and task queue persistence.",
    tech_stack  = ["Python", "Ollama", "ChromaDB"],
    category    = "AI Agents",
    highlights  = ["Handles 5 operations autonomously", "Approval gate for destructive ops"],
)
_P3 = ProjectEntry(
    name        = "RAG Chatbot",
    tagline     = "Q&A chatbot grounded in your documents.",
    description = "Retrieval-augmented generation over a personal knowledge base.",
    tech_stack  = ["Python", "ChromaDB", "Ollama", "FastAPI"],
    github_url  = "https://github.com/testuser/rag-chatbot",
    demo_url    = "https://rag-chatbot.example.com",
    category    = "Text AI",
)

_CFG = PortfolioConfig(
    owner_name      = "Jane Doe",
    title           = "AI Engineer",
    bio             = "I build practical AI applications with Python. "
                      "100 days of AI engineering, shipped.",
    email           = "jane@example.com",
    github_username = "janedoe",
    linkedin_url    = "https://linkedin.com/in/janedoe",
    projects        = [_P1, _P2, _P3],
)

def generate_case_study(project):
    """Generate a Markdown case study for one project.

    Structure:
      # {project.name}
      **Category:** {project.category}
      ## Overview
      {project.tagline}
      {project.description}
      ## Tech Stack
      - item 1
      - item 2
      ## Key Achievements
      - highlight 1 (or placeholder if highlights is empty)
      ## Links
      - GitHub: {url}
      - Demo: {url}   (only if demo_url is set)

    Returns:
        str — Markdown starting with "# {project.name}"
    """
    tech_list  = "\n".join(f"- {t}" for t in project.tech_stack)
    highlights = (
        "\n".join(f"- {h}" for h in project.highlights)
        if project.highlights else "- See project README for details"
    )
    links = []
    if project.github_url: links.append(f"- GitHub: {project.github_url}")
    if project.demo_url:   links.append(f"- Demo: {project.demo_url}")
    links_text = "\n".join(links) if links else "- See GitHub profile"
    # TODO: assemble and return the Markdown string
    return ""


### Checks

In [ ]:
checks = 0

# 1 — starts with # project.name
try:
    md = generate_case_study(_P1)
    assert md.startswith(f"# {_P1.name}"),         f"expected '# {_P1.name}', got: {md[:40]!r}"
    checks += 1; print(f"✅ 1 case study starts with '# {_P1.name}'")
except Exception as e:
    print("❌ 1:", e)

# 2 — required ## sections present
try:
    md = generate_case_study(_P1)
    for section in ["## Overview", "## Tech Stack", "## Key Achievements", "## Links"]:
        assert section in md, f"missing {section}"
    checks += 1; print("✅ 2 all ## sections present (Overview, Tech Stack, Achievements, Links)")
except Exception as e:
    print("❌ 2:", e)

# 3 — tech stack items appear as bullet points
try:
    md = generate_case_study(_P1)
    for tech in _P1.tech_stack:
        assert f"- {tech}" in md, f"tech {tech!r} not as bullet point"
    checks += 1; print(f"✅ 3 all {len(_P1.tech_stack)} tech stack items as bullet points")
except Exception as e:
    print("❌ 3:", e)

# 4 — highlights appear; empty highlights → placeholder
try:
    md_with = generate_case_study(_P1)
    for h in _P1.highlights:
        assert h in md_with, f"highlight {h!r} not in case study"
    p_no_highlights = _P1.__class__(
        name="X", tagline="T", description="D", tech_stack=["py"]
    )
    md_empty = generate_case_study(p_no_highlights)
    assert "See project README" in md_empty, "empty highlights should have placeholder"
    checks += 1; print("✅ 4 highlights present; empty highlights → placeholder text")
except Exception as e:
    print("❌ 4:", e)

# 5 — GitHub URL present; demo_url included only when set
try:
    md = generate_case_study(_P1)      # has github_url, no demo
    assert _P1.github_url in md,       f"github_url not in links"
    assert "Demo:" not in md,          "demo URL should not appear (_P1 has no demo)"
    md3 = generate_case_study(_P3)     # has both github_url and demo_url
    assert _P3.demo_url in md3,        "demo_url not in _P3 case study"
    checks += 1; print("✅ 5 GitHub link present; Demo link only when demo_url is set")
except Exception as e:
    print("❌ 5:", e)

print(f"\n{checks}/5 checks passed!")
